In [ ]:
import numpy as np
import open3d as o3d
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("./coche_coche.csv")

points = df[["x", "y", "z"]].to_numpy(dtype=np.float64)

points = points[np.linalg.norm(points, axis=1) > 0.1]

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(points)

pcd_down = pcd.voxel_down_sample(0.08)

plane_model, ground_indices = pcd_down.segment_plane(
    distance_threshold=0.15,
    ransac_n=3,
    num_iterations=100
)

[a, b, c, d] = plane_model

ground = pcd_down.select_by_index(ground_indices)
objects = pcd_down.select_by_index(ground_indices, invert=True)

print("Puntos originales:", len(points))
print("Puntos reducidos:", len(pcd_down.points))
print("Puntos carretera:", len(ground.points))
print("Puntos objetos:", len(objects.points))

labels = np.array(
    objects.cluster_dbscan(
        eps=0.35,
        min_points=8,
        print_progress=True
    )
)

max_label = labels.max()

print("Clusters encontrados:", max_label + 1)

for i in range(max_label + 1):
    cantidad = np.sum(labels == i)
    print("Cluster", i, ":", cantidad, "puntos")

ground.paint_uniform_color([0.5, 0.5, 0.5])

colors = np.zeros((len(objects.points), 3))

if max_label >= 0:
    cluster_colors = plt.get_cmap("tab20")(
        labels / max_label
    )[:, :3]

    colors[labels >= 0] = cluster_colors[labels >= 0]

    colors[labels < 0] = [0.2, 0.2, 0.2]

objects.colors = o3d.utility.Vector3dVector(colors)

o3d.visualization.draw_geometries(
    [ground, objects],
    zoom=0.5,
    front=[-0.4999, -0.1659, -0.8499],
    lookat=[2.1813, 2.0619, 2.0999],
    up=[0.1204, -0.9852, 0.1215]
)